# Raster resampling: nearest neighbour and bilinear interpolation

This lab changes the raster shape while keeping the same extent and orientation. It is not a CRS-aware geospatial warp.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'implementations').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'implementations/python/raster-resampling'))
from resampling import resample_to_shape


## 1. Categorical values: nearest neighbour

Nearest neighbour copies a source class code. It does not create new categories by averaging numeric labels.

In [ ]:
classes=np.array([[1,1,2,2],[1,3,3,2],[4,3,3,2],[4,4,4,2]])
nearest=resample_to_shape(classes,(12,12),method='nearest')
fig,axes=plt.subplots(1,2,figsize=(8,4))
for ax,data,title in zip(axes,[classes,nearest],['Source classes','Nearest neighbour']):
    im=ax.imshow(data,cmap='tab10',vmin=1,vmax=4); ax.set_title(title); fig.colorbar(im,ax=ax,ticks=[1,2,3,4])
plt.tight_layout(); plt.show()
print('Output classes:',np.unique(nearest))


## 2. Continuous values: bilinear interpolation

Bilinear interpolation combines the four surrounding samples. A plane is a useful test surface because bilinear interpolation should reproduce its value at mapped interior positions.

In [ ]:
surface=np.fromfunction(lambda r,c:2*r+3*c,(4,4),dtype=float)
bilinear=resample_to_shape(surface,(7,7),method='bilinear')
fig,axes=plt.subplots(1,2,figsize=(8,4))
for ax,data,title in zip(axes,[surface,bilinear],['Source plane','Bilinear output']):
    im=ax.imshow(data,cmap='viridis'); ax.set_title(title); fig.colorbar(im,ax=ax)
plt.tight_layout(); plt.show()
print('Mapped centre value:',bilinear[3,3])


## 3. NoData policy

Strict mode masks a target when a positive-weight source neighbour is missing. Renormalization averages the remaining valid neighbours.

In [ ]:
gappy=surface.copy(); gappy[1,1]=np.nan
strict=resample_to_shape(gappy,(7,7),method='bilinear',nodata_policy='strict')
renorm=resample_to_shape(gappy,(7,7),method='bilinear',nodata_policy='renormalize')
fig,axes=plt.subplots(1,2,figsize=(8,4))
for ax,data,title in zip(axes,[strict,renorm],['Strict NoData','Renormalized valid neighbours']):
    im=ax.imshow(data,cmap='viridis'); ax.set_title(title); fig.colorbar(im,ax=ax)
plt.tight_layout(); plt.show()


## Investigate

1. Why is nearest neighbour appropriate for categorical data?
2. Why does bilinear interpolation smooth a continuous surface?
3. Compare strict and renormalized NoData policies.
4. Why does a production warp need the CRS and affine transform?
5. When might area-weighted aggregation be more appropriate than interpolation?